# Data Cleaning and Labeling

In this notebook, I clean the raw Amazon review dataset, handle missing values, and construct sentiment labels based on review ratings.

The goal is to prepare a high-quality dataset for further analysis and modeling.

In [8]:
import pandas as pd
import numpy as np

In [9]:
df = pd.read_csv("../data/raw/amazon_sample.csv")
df.head()

C:\Users\ASUS\AppData\Local\Temp\ipykernel_32844\2374893580.py:1: DtypeWarning: Columns (0: vote) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/raw/amazon_sample.csv")


,overall,vote,verified,reviewTime,reviewerID,asin,style,reviewerName,reviewText,summary,unixReviewTime,image
0,5.0,67,True,"09 18, 1999",AAP7PPBU72QFM,0151004714,{'Format:': ' Hardcover'},D. C. Carrad,This is the best novel I have read in 2 or 3 y...,A star is born,937612800,NaN
1,3.0,5,True,"10 23, 2013",A2E168DTVGE6SV,0151004714,{'Format:': ' Kindle Edition'},Evy,"Pages and pages of introspection, in the style...",A stream of consciousness novel,1382486400,NaN
2,5.0,4,False,"09 2, 2008",A1ER5AYS3FQ9O3,0151004714,{'Format:': ' Paperback'},Kcorn,This is the kind of novel to read when you hav...,I'm a huge fan of the author and this one did ...,1220313600,NaN
3,5.0,13,False,"09 4, 2000",A1T17LMQABMBN5,0151004714,{'Format:': ' Hardcover'},Caf Girl Writes,What gorgeous language! What an incredible wri...,The most beautiful book I have ever read!,968025600,NaN
4,3.0,8,True,"02 4, 2000",A3QHJ0FXK33OBE,0151004714,{'Format:': ' Hardcover'},W. Shane Schmidt,I was taken in by reviews that compared this b...,A dissenting view--In part.,949622400,NaN


In [10]:
df.shape
df.columns
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100001 entries, 0 to 100000
Data columns (total 12 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   overall         100001 non-null  float64
 1   vote            17157 non-null   object 
 2   verified        100001 non-null  bool   
 3   reviewTime      100001 non-null  str    
 4   reviewerID      100001 non-null  str    
 5   asin            100001 non-null  str    
 6   style           61751 non-null   str    
 7   reviewerName    99976 non-null   str    
 8   reviewText      99982 non-null   str    
 9   summary         99993 non-null   str    
 10  unixReviewTime  100001 non-null  int64  
 11  image           1000 non-null    str    
dtypes: bool(1), float64(1), int64(1), object(1), str(8)
memory usage: 8.5+ MB


In [11]:
columns_to_keep = [
    "overall",
    "vote",
    "verified",
    "reviewTime",
    "reviewText",
    "summary",
    "unixReviewTime"
]

df = df[columns_to_keep].copy()
df.head()

,overall,vote,verified,reviewTime,reviewText,summary,unixReviewTime
0,5.0,67,True,"09 18, 1999",This is the best novel I have read in 2 or 3 y...,A star is born,937612800
1,3.0,5,True,"10 23, 2013","Pages and pages of introspection, in the style...",A stream of consciousness novel,1382486400
2,5.0,4,False,"09 2, 2008",This is the kind of novel to read when you hav...,I'm a huge fan of the author and this one did ...,1220313600
3,5.0,13,False,"09 4, 2000",What gorgeous language! What an incredible wri...,The most beautiful book I have ever read!,968025600
4,3.0,8,True,"02 4, 2000",I was taken in by reviews that compared this b...,A dissenting view--In part.,949622400


In [12]:
df.isnull().sum()

overall               0
vote              82844
verified              0
reviewTime            0
reviewText           19
summary               8
unixReviewTime        0
dtype: int64

In [13]:
df = df.dropna(subset=["overall", "reviewText"]).copy()
df.shape

(99982, 7)

In [14]:
df["vote"] = df["vote"].fillna("0").astype(str).str.replace(",", "", regex=False)
df["vote"] = pd.to_numeric(df["vote"], errors="coerce").fillna(0).astype(int)

df["vote"].describe()

count    99982.000000
mean         2.246374
std         16.720958
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max       1700.000000
Name: vote, dtype: float64

In [15]:
df["verified"] = df["verified"].fillna(False).astype(bool)
df["verified"].value_counts()

verified
True     85964
False    14018
Name: count, dtype: int64

In [16]:
df["summary"] = df["summary"].fillna("")

## Label Construction

Sentiment is inferred from the original star rating:
- 4–5 stars → positive
- 1–2 stars → negative
- 3 stars → neutral

For binary sentiment classification, 3-star reviews are removed because they are often ambiguous and may introduce label noise.

In [17]:
def label_sentiment(rating):
    if rating >= 4:
        return "positive"
    elif rating <= 2:
        return "negative"
    else:
        return "neutral"

df["sentiment"] = df["overall"].apply(label_sentiment)
df["sentiment"].value_counts()

sentiment
positive    84287
negative     8807
neutral      6888
Name: count, dtype: int64

In [18]:
df = df[df["sentiment"] != "neutral"].copy()
df["sentiment"].value_counts()

sentiment
positive    84287
negative     8807
Name: count, dtype: int64

In [ ]:
df["review_length"] = df["reviewText"].astype(str).apply(len)
df["word_count"] = df["reviewText"].astype(str).apply(lambda x: len(x.split()))
df[["review_length", "word_count"]].describe()

,review_length,word_count
count,93094.000000,93094.000000
mean,321.476927,59.017262
std,532.680363,95.935476
min,1.000000,1.000000
25%,52.000000,10.000000
50%,149.000000,28.000000
75%,362.000000,67.000000
max,19727.000000,3624.000000


In [20]:
df.head()
df.shape
df["sentiment"].value_counts(normalize=True)

sentiment
positive    0.905397
negative    0.094603
Name: proportion, dtype: float64

## Class Distribution

The dataset is highly imbalanced, with approximately 90% positive reviews and only 10% negative reviews.

This imbalance is expected in real-world e-commerce data, where satisfied customers are more likely to leave reviews.

However, this poses a challenge for classification models, as they may become biased toward predicting the majority class.

In [21]:
df.to_csv("../data/processed/amazon_sentiment_processed.csv", index=False)